In [1]:
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependenc

In [4]:
import os
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


#INGESTION (MEMUAT DATA)

In [7]:
!wget -O buku_panduan_gen_ai.pdf "https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb"

--2026-05-21 03:18:52--  https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.google.com (drive.google.com)... 74.125.142.139, 74.125.142.100, 74.125.142.102, ...
Connecting to drive.google.com (drive.google.com)|74.125.142.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb [following]
--2026-05-21 03:18:52--  https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 173.194.202.132, 2607:f8b0:400e:c00::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|173.194.202.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 655208 (640K) [application/octet-stream]
Saving to: ‘buku_panduan_gen_ai.pdf’

buku_panduan_gen_ai 100%[===================>] 639.85K  --.-KB/s    in 0.005s  

2026-05-21 03:18:54 (

In [8]:
file_path = "/content/buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

In [9]:
documents[0]

Document(metadata={'producer': 'iLovePDF', 'creator': 'Adobe InDesign 20.3 (Windows)', 'creationdate': '2025-06-03T13:36:46+07:00', 'source': '/content/buku_panduan_gen_ai.pdf', 'file_path': '/content/buku_panduan_gen_ai.pdf', 'total_pages': 42, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-06-04T05:01:35+00:00', 'trapped': '', 'modDate': 'D:20250604050135Z', 'creationDate': "D:20250603133646+07'00'", 'page': 0}, page_content='')

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    separators=["\n\n", "\n", "."]
    )

#Proses split dokumen
chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)} chunks")

Total chunks created: 78chunks


#STORING (MENYIMPAN DATA)

In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", 
    model_kwargs={"device": device}
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [12]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

#RETRIEVER (MENGAMBIL DATA)

In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
    )

In [14]:
query = "Apa itu Generative AI?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"---- Document {i} ----")
    print(doc.page_content)
    print()

---- Document 1 ----
8
BAB II: Pemahaman Teknologi Generative AI
2.1. Perbedaan AI dan Generative AI
Sebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk 
memahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-
cara khusus.
1.	 Artificial Intelligence (AI)
•	 AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-
agai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.
•	 Contoh AI meliputi asisten virtual seperti Siri dan Google Assistant, algoritma pen-
carian Google, dan sistem rekomendasi Netflix atau Spotify.
•	 AI umumnya bersifat berbasis aturan atau pembelajaran mesin yang membantu ma-
nusia dalam mengambil keputusan.
2. Generative AI (GenAI)
•	 GenAI adalah cabang dari AI yang mampu menghasilkan konten baru, seperti teks, 
gambar, video, dan suara, berdasarkan data yang telah dipelajarinya.
•	 Model GenAI menggunakan teknik deep learning dan neural networks untuk men-

---- Documen

#GENERATION (TEKS GENERATE)

In [15]:
#Konfigurasi Kuantizasi untuk model 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [16]:
model_name = "unsloth/Llama-3.2-3B-Instruct"

#Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

#Load Model dengan konfigurasi kuantizasi
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [17]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0


In [18]:
from langchain_core.prompts import PromptTemplate

#Menulis template prompt untuk RAG dengan format Llama-3.2-3B-Instruct
template ="""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Anda adalah asisten AI yang bertugas membantu pengguna.
GUnakan Hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jjur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan.
Berikan jawaban secara singkat dan jelas.

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

In [19]:
#Membuat RAG Chain
rag_chain = (
    {"context": retriever, "query": RunnablePassthrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)

#TESTING MODEL RAG

In [22]:
def ask_question(query):
    print(f"Pertanyaan: {query}\n")
    
    #Menjalankan RAG Chain dengan query
    response = rag_chain.invoke(query)
    
    print("Jawaban")
    print(response)
    
    #Tampilkan dokumen yang diretriev untuk referensi
    docs = retriever.invoke(query)
    print("\nDokumen Referensi:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. Halaman {doc.metadata.get('page', '?')}")

In [23]:
query = "Apa itu Generative AI?"
ask_question(query)

Pertanyaan: Apa itu Generative AI?

Jawaban
Generative AI (GenAI) adalah cabang dari Artificial Intelligence (AI) yang dirancang untuk menghasilkan konten baru, seperti teks, gambar, suara, dan video, berdasarkan data yang telah dipelajarinya.

Dokumen Referensi:
1. Halaman 8
2. Halaman 38
3. Halaman 9
